# 📊 ShopBR Marketplace — EDA
## Análise Exploratória dos Dados de Vendas

**Objetivo:** Compreender o comportamento histórico de vendas por categoria para embasar a estratégia de modelagem.

**Dataset:** 2022-01-01 a 2024-03-31 · 5 categorias · frequência diária

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
PALETTE = ['#00e5ff','#ff5e57','#a8ff3e','#a29bfe','#ff9f43']
plt.rcParams.update({'figure.facecolor':'#0e1420','axes.facecolor':'#080c14',
                     'axes.edgecolor':'#1c2438','grid.color':'#1c2438',
                     'text.color':'#e2eaf4','axes.labelcolor':'#e2eaf4',
                     'xtick.color':'#5a6a80','ytick.color':'#5a6a80','figure.dpi':120})
print('Setup OK')

In [ ]:
df = pd.read_csv('../data/raw/vendas.csv', parse_dates=['data'])
print(f'Shape: {df.shape}')
print(f'Período: {df.data.min().date()} → {df.data.max().date()}')
print(f'Categorias: {df.categoria.unique().tolist()}')
df.head(10)

## 1. Estatísticas Descritivas

In [ ]:
resumo = df.groupby('categoria').agg(
    dias=('data','count'), uni_med=('unidades','mean'), uni_std=('unidades','std'),
    uni_max=('unidades','max'), rec_total=('receita','sum')
).round(1)
resumo['rec_total'] = (resumo['rec_total']/1e6).round(2)
resumo.columns = ['Dias','Média/dia','Desvio','Máximo','Receita (R$ M)']
resumo

## 2. Série Temporal por Categoria

In [ ]:
cats = df['categoria'].unique()
fig, axes = plt.subplots(5, 1, figsize=(14,14), sharex=True)
for ax, cat, cor in zip(axes, cats, PALETTE):
    s = df[df['categoria']==cat].set_index('data')['unidades'].rolling(7).mean()
    ax.plot(s.index, s.values, color=cor, linewidth=1.2)
    ax.fill_between(s.index, s.values, alpha=0.08, color=cor)
    ax.set_ylabel(cat, fontsize=9)
    ax.grid(True, alpha=0.3)
fig.suptitle('Vendas Diárias por Categoria (MA-7)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Sazonalidade Mensal

In [ ]:
df['mes'] = df['data'].dt.month
piv = df.groupby(['mes','categoria'])['unidades'].mean().unstack()
fig, ax = plt.subplots(figsize=(12,4))
for cat, cor in zip(piv.columns, PALETTE):
    ax.plot(piv.index, piv[cat], marker='o', color=cor, linewidth=2, label=cat, markersize=5)
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'])
ax.set_ylabel('Média Unidades/Dia')
ax.set_title('Sazonalidade Mensal por Categoria')
ax.legend()
ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Sazonalidade por Dia da Semana

In [ ]:
df['dow'] = df['data'].dt.dayofweek
dow_labels = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
piv_dow = df.groupby(['dow','categoria'])['unidades'].mean().unstack()
fig, ax = plt.subplots(figsize=(10,4))
x = np.arange(7); w = 0.15
for i,(cat,cor) in enumerate(zip(piv_dow.columns, PALETTE)):
    ax.bar(x+i*w, piv_dow[cat], width=w, color=cor, label=cat, alpha=0.85)
ax.set_xticks(x+w*2); ax.set_xticklabels(dow_labels)
ax.set_title('Sazonalidade por Dia da Semana')
ax.legend(fontsize=9); ax.grid(True,alpha=0.3,axis='y')
plt.tight_layout(); plt.show()

## 5. Correlação entre Categorias

In [ ]:
pivot = df.pivot_table(index='data', columns='categoria', values='unidades')
corr = pivot.corr()
fig, ax = plt.subplots(figsize=(7,5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, annot=True, fmt='.2f', ax=ax)
ax.set_title('Correlação de Pearson entre Categorias')
plt.tight_layout(); plt.show()

## 6. Análise do Black Friday

In [ ]:
nov = df[df['data'].dt.month==11].copy()
fig, ax = plt.subplots(figsize=(13,4))
for cat,cor in zip(cats, PALETTE):
    s = nov[nov['categoria']==cat].set_index('data')['unidades']
    ax.plot(s.index, s.values, color=cor, linewidth=1.5, label=cat)
ax.axvspan(pd.Timestamp('2022-11-25'), pd.Timestamp('2022-11-27'), alpha=0.15, color='yellow')
ax.axvspan(pd.Timestamp('2023-11-24'), pd.Timestamp('2023-11-26'), alpha=0.15, color='orange')
ax.set_title('Novembro: Impacto do Black Friday por Categoria')
ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

## Conclusões
- **Black Friday (Nov)** gera pico de ~2.8x sobre a média em todas as categorias
- **Dezembro** é o 2º maior mês (Natal + presentes)
- **Janeiro–Fevereiro** são os meses mais fracos (~30% abaixo da média)
- Fins de semana têm ~20-25% mais vendas que dias úteis
- **Moda e Beleza** mostram crescimento de tendência mais acelerado
- Baixa correlação entre categorias → comportamentos de compra independentes

→ **Próximo notebook:** Feature Engineering